# Step 2b — ElasticNet Convergence Fix Validation
**Purpose:** The previous step2 run used `ElasticNet(max_iter=200)` on unscaled freight-rate targets.  
Duality gaps were ~5,000× over convergence tolerance. This notebook re-runs the full 5-fold  
walk-forward with a properly converged ElasticNet (`max_iter=5000`, y StandardScaled before fit).  

**Key question:** Does `supramax_7d` remain PROMOTE, or was that verdict an artefact of the convergence bug?


In [ ]:
import os
REPO = "https://github.com/SSOHEB/FICOS-Platform.git"
if not os.path.exists("FICOS-Platform"):
    !git clone {REPO}
os.chdir("FICOS-Platform")
print("Working dir:", os.getcwd())


In [ ]:
!pip install -q lightgbm xgboost scikit-learn pandas numpy
print("Packages ready.")


In [ ]:
import pandas as pd, numpy as np, warnings
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge, ElasticNet
from sklearn.ensemble import RandomForestRegressor
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.base import clone
import xgboost as xgb, lightgbm as lgb

warnings.filterwarnings('ignore')

# ── Load data ──────────────────────────────────────────────────────────────────
df = pd.read_csv('outputs/modeling_dataset.csv')
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date').reset_index(drop=True)
n_rows = len(df)

all_cols     = list(df.columns)
leakage_cols = [c for c in all_cols if c.startswith('dir_') or
                c.startswith('future_') or c.startswith('target_')]
drop_cols    = set(leakage_cols + ['date'])
feature_cols = [c for c in all_cols if c not in drop_cols]
print(f"Dataset: {n_rows} rows | {len(feature_cols)} clean features")

# ── Fold definitions (identical to step2) ─────────────────────────────────────
test_size, val_size = 250, 200
fold_configs = {}
for f in range(1, 6):
    te_end   = n_rows - (5 - f) * test_size
    te_start = te_end - test_size
    va_end, va_start = te_start, te_start - val_size
    fold_configs[f] = dict(tr_end=va_start, va_start=va_start, va_end=va_end,
                           te_start=te_start, te_end=te_end)
    tr_d = df.date.iloc[0].strftime('%Y-%m-%d')
    va_d = df.date.iloc[va_start].strftime('%Y-%m-%d')
    te_d = df.date.iloc[te_start].strftime('%Y-%m-%d')
    print(f"  Fold {f}: train ends row {va_start} | val {va_d} | test {te_d}")

assets   = ['cape', 'panamax', 'supramax', 'handy', 'kdci']
horizons = [1, 7, 14, 30]

# ── Metric helpers ─────────────────────────────────────────────────────────────
def smape(a, b):
    return float(np.mean(200*np.abs(b-a)/(np.abs(a)+np.abs(b)+1e-8)))
def r2(a, b):
    ss = np.sum((a-np.mean(a))**2)
    return float(1 - np.sum((a-b)**2)/ss if ss > 0 else np.nan)
def uda(a, b):
    m = (a!=0)&(b!=0)
    return 50.0 if m.sum()==0 else float(np.mean(np.sign(a[m])==np.sign(b[m]))*100)

# ── Main walk-forward loop ─────────────────────────────────────────────────────
wf_records = []

for fold_id in range(1, 6):
    print(f"\n── Fold {fold_id}/5 ──", flush=True)
    cfg = fold_configs[fold_id]
    tr_end, va_start, va_end, te_start, te_end = (
        cfg['tr_end'], cfg['va_start'], cfg['va_end'], cfg['te_start'], cfg['te_end'])

    for asset in assets:
        for h in horizons:
            pair_key   = f'{asset}_{h}d'
            df_p       = df.copy()
            df_p['_yd'] = df_p[asset].shift(-h) - df_p[asset]
            df_v       = df_p[~df_p['_yd'].isna()].reset_index(drop=True)

            tr_m = np.zeros(len(df_v), bool); tr_m[:tr_end]         = True
            va_m = np.zeros(len(df_v), bool); va_m[va_start:va_end] = True
            te_m = np.zeros(len(df_v), bool); te_m[te_start:te_end] = True

            X_raw  = df_v[feature_cols].values.copy()
            y_d    = df_v['_yd'].values
            y_base = df_v[asset].values

            med = np.nanmedian(X_raw[tr_m], axis=0)
            med[np.isnan(med)] = 0.0
            for ci in range(X_raw.shape[1]):
                X_raw[:, ci] = np.where(np.isnan(X_raw[:, ci]), med[ci], X_raw[:, ci])

            sx  = StandardScaler()
            Xtr = sx.fit_transform(X_raw[tr_m])
            Xva = sx.transform(X_raw[va_m])
            Xte = sx.transform(X_raw[te_m])

            sy     = StandardScaler()
            ytr_sc = sy.fit_transform(y_d[tr_m].reshape(-1,1)).flatten()

            sel   = SelectKBest(f_regression, k=30)
            Xtr_s = sel.fit_transform(Xtr, y_d[tr_m])
            Xva_s = sel.transform(Xva)
            Xte_s = sel.transform(Xte)

            models = {}  # key -> (model, val_pred, te_pred)

            # Ridge
            br, brv = None, float('inf')
            for a in [0.1, 1.0, 10.0, 100.0, 1000.0]:
                m = Ridge(alpha=a).fit(Xtr_s, y_d[tr_m])
                s = smape(y_d[va_m], m.predict(Xva_s))
                if s < brv: brv, br = s, m
            models['Ridge'] = (br, br.predict(Xva_s), br.predict(Xte_s))

            # ElasticNet FIXED: max_iter=5000 + scaled y
            be, bev, en_va, en_te = None, float('inf'), None, None
            for a in [0.01, 0.1, 1.0]:
                for l1 in [0.2, 0.5, 0.8]:
                    m    = ElasticNet(alpha=a, l1_ratio=l1, max_iter=5000,
                                      random_state=42).fit(Xtr_s, ytr_sc)
                    va_p = sy.inverse_transform(m.predict(Xva_s).reshape(-1,1)).flatten()
                    s    = smape(y_d[va_m], va_p)
                    if s < bev:
                        bev, be = s, m
                        en_va   = va_p
                        en_te   = sy.inverse_transform(m.predict(Xte_s).reshape(-1,1)).flatten()
            models['ElasticNet'] = (be, en_va, en_te)

            # RandomForest
            brf, brfv = None, float('inf')
            for ne in [50, 100]:
                for d in [3, 5]:
                    m = RandomForestRegressor(n_estimators=ne, max_depth=d,
                                              random_state=42, n_jobs=-1).fit(Xtr_s, y_d[tr_m])
                    s = smape(y_d[va_m], m.predict(Xva_s))
                    if s < brfv: brfv, brf = s, m
            models['RandomForest'] = (brf, brf.predict(Xva_s), brf.predict(Xte_s))

            # XGBoost
            bxg, bxgv = None, float('inf')
            for ne in [50, 100]:
                for d in [3, 4]:
                    m = xgb.XGBRegressor(n_estimators=ne, max_depth=d, learning_rate=0.05,
                                         random_state=42, n_jobs=-1).fit(Xtr_s, y_d[tr_m])
                    s = smape(y_d[va_m], m.predict(Xva_s))
                    if s < bxgv: bxgv, bxg = s, m
            models['XGBoost'] = (bxg, bxg.predict(Xva_s), bxg.predict(Xte_s))

            # LightGBM
            blg, blgv = None, float('inf')
            for ne in [50, 100]:
                for d in [3, 4]:
                    m = lgb.LGBMRegressor(n_estimators=ne, max_depth=d, learning_rate=0.05,
                                          random_state=42, verbose=-1, n_jobs=-1).fit(Xtr_s, y_d[tr_m])
                    s = smape(y_d[va_m], m.predict(Xva_s))
                    if s < blgv: blgv, blg = s, m
            models['LightGBM'] = (blg, blg.predict(Xva_s), blg.predict(Xte_s))

            winner  = min(models, key=lambda k: smape(y_d[va_m], models[k][1]))
            w_va_p  = models[winner][1]
            pred_te = models[winner][2]
            w_model = models[winner][0]

            te_r2v = r2(y_d[te_m], pred_te)
            te_dav = uda(y_d[te_m], pred_te)

            val_resids = y_d[va_m] - w_va_p
            p10, p90   = np.percentile(val_resids, 10), np.percentile(val_resids, 90)
            pct_p      = pred_te / (np.abs(y_base[te_m]) + 1e-8)
            buy_m      = (pred_te > max(0.0, p90)) & (pct_p >  0.01)
            wait_m     = (pred_te < min(0.0, p10)) & (pct_p < -0.01)
            n_fired    = int((buy_m | wait_m).sum())

            perm_das = []
            rng = np.random.RandomState(42)
            for _ in range(20):
                yp  = rng.permutation(y_d[tr_m])
                sp  = SelectKBest(f_regression, k=30)
                Xp  = sp.fit_transform(Xtr, yp)
                Xtp = sp.transform(Xte)
                mp  = clone(w_model)
                if winner == 'ElasticNet':
                    yp_sc = sy.fit_transform(yp.reshape(-1,1)).flatten()
                    mp.fit(Xp, yp_sc)
                    pp = sy.inverse_transform(mp.predict(Xtp).reshape(-1,1)).flatten()
                else:
                    mp.fit(Xp, yp)
                    pp = mp.predict(Xtp)
                perm_das.append(uda(y_d[te_m], pp))

            perm_max = float(np.max(perm_das))

            if   n_fired < 15:               v = 'INSUFFICIENT'
            elif te_dav <= perm_max:         v = 'FAILS PERM'
            elif te_r2v <= 0 or te_dav < 50: v = 'UNDERFIT'
            else:                            v = 'GENUINE'

            wf_records.append({'pair': pair_key, 'fold': fold_id, 'winner': winner,
                                'te_r2': round(te_r2v,4), 'te_da': round(te_dav,1),
                                'perm_max': round(perm_max,1), 'n_fired': n_fired, 'verdict': v})

print("\nAll folds complete.")


In [ ]:
import os
os.makedirs('outputs', exist_ok=True)

df_all = pd.DataFrame(wf_records)
df_all.to_csv('outputs/step2b_elasticnet_fixed_detail.csv', index=False)

summary = []
for p in df_all['pair'].unique():
    sub   = df_all[df_all['pair'] == p]
    g_cnt = (sub['verdict'] == 'GENUINE').sum()
    summary.append({'pair': p, 'genuine_folds': f'{g_cnt}/5',
                    'dominant_winner': sub['winner'].mode()[0],
                    'recommendation': 'PROMOTE' if g_cnt >= 3 else 'EXCLUDE'})

df_sum = pd.DataFrame(summary)
df_sum.to_csv('outputs/step2b_elasticnet_fixed_summary.csv', index=False)

print("="*70)
print("STEP 2b RESULT — ElasticNet FIXED (max_iter=5000, y scaled)")
print("="*70)
promoted = df_sum[df_sum['recommendation']=='PROMOTE']
print(f"\nPROMOTED PAIRS ({len(promoted)} total):")
print(promoted.to_string(index=False) if len(promoted) else "  (none)")
print("\nFULL 20-PAIR SUMMARY:")
print(df_sum.to_string(index=False))
print("\nSaved: outputs/step2b_elasticnet_fixed_summary.csv")
print("       outputs/step2b_elasticnet_fixed_detail.csv")
